# Unificación y auditoría de datos 2024-2025

Este notebook prepara la información para la etapa posterior de selección de target y modelado.

Alcance de este notebook:

- Cargar `tabla_modelo_2024.csv` y `tabla_modelo_2025.csv`.
- Comparar columnas, llaves, tipos, cobertura, valores representativos y códigos especiales.
- Generar un diccionario técnico inicial de variables.
- Crear una unión bruta en memoria solamente si ambas estructuras son compatibles.

No decide el target, no imputa faltantes, no elimina registros y no crea variables nuevas.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

# Ubica automáticamente la raíz del proyecto aunque el notebook se ejecute desde modelo/.
RUTA_ACTUAL = Path.cwd().resolve()
RAIZ = next(
    ruta for ruta in [RUTA_ACTUAL, *RUTA_ACTUAL.parents]
    if (ruta / 'datos_procesados').exists()
)

RUTA_DATOS = RAIZ / 'datos_procesados'
RUTA_2024 = RUTA_DATOS / 'tabla_modelo_2024.csv'
RUTA_2025 = RUTA_DATOS / 'tabla_modelo_2025.csv'

for ruta in [RUTA_2024, RUTA_2025]:
    if not ruta.exists():
        raise FileNotFoundError(f'No se encontró el archivo: {ruta}')

print(f'Raíz del proyecto: {RAIZ}')
print(f'Datos 2024: {RUTA_2024.name}')
print(f'Datos 2025: {RUTA_2025.name}')

In [ ]:
# Los identificadores se cargan como texto para preservar su formato.
COLUMNAS_ID = {'caseid': 'string', 'hhid': 'string', 'idx_nino': 'string'}

def cargar_tabla_modelo(ruta):
    return pd.read_csv(
        ruta,
        dtype=COLUMNAS_ID,
        low_memory=False
    )

tabla_modelo_2024 = cargar_tabla_modelo(RUTA_2024)
tabla_modelo_2025 = cargar_tabla_modelo(RUTA_2025)

print(f'2024: {tabla_modelo_2024.shape[0]:,} filas × {tabla_modelo_2024.shape[1]} columnas')
print(f'2025: {tabla_modelo_2025.shape[0]:,} filas × {tabla_modelo_2025.shape[1]} columnas')

## 1. Estructura de columnas

La unión solo se realizará si los dos años tienen el mismo conjunto de columnas. Las diferencias deben revisarse antes de continuar.

In [ ]:
columnas_2024 = set(tabla_modelo_2024.columns)
columnas_2025 = set(tabla_modelo_2025.columns)

comparacion_columnas = pd.DataFrame({
    'variable': sorted(columnas_2024 | columnas_2025)
})
comparacion_columnas['en_2024'] = comparacion_columnas['variable'].isin(columnas_2024)
comparacion_columnas['en_2025'] = comparacion_columnas['variable'].isin(columnas_2025)
comparacion_columnas['estado'] = 'comun'
comparacion_columnas.loc[
    comparacion_columnas['en_2024'] & ~comparacion_columnas['en_2025'],
    'estado'
] = 'solo_2024'
comparacion_columnas.loc[
    ~comparacion_columnas['en_2024'] & comparacion_columnas['en_2025'],
    'estado'
] = 'solo_2025'

display(comparacion_columnas['estado'].value_counts().rename_axis('estado').to_frame('n_variables'))
display(comparacion_columnas.query("estado != 'comun'"))

## 2. Auditoría de llaves

La llave esperada de un niño dentro de cada año es `caseid + idx_nino`. Se incorpora `anio` al comparar ambos años.

In [ ]:
LLAVES_NINO = ['caseid', 'idx_nino']
COLUMNAS_REQUERIDAS = ['anio', 'caseid', 'idx_nino', 'hhid']

def auditar_llaves(tabla, anio_esperado):
    faltantes = set(COLUMNAS_REQUERIDAS) - set(tabla.columns)
    if faltantes:
        raise KeyError(f'Faltan columnas requeridas: {sorted(faltantes)}')

    llaves = tabla[LLAVES_NINO].copy()
    for columna in LLAVES_NINO:
        llaves[columna] = llaves[columna].astype('string').str.strip()

    return {
        'anio_revisado': anio_esperado,
        'filas': len(tabla),
        'anio_registrado': ', '.join(map(str, sorted(tabla['anio'].dropna().unique()))),
        'caseid_faltante': llaves['caseid'].isna().sum(),
        'idx_nino_faltante': llaves['idx_nino'].isna().sum(),
        'llaves_nino_duplicadas': llaves.duplicated(LLAVES_NINO).sum(),
        'hogares_unicos': tabla['hhid'].astype('string').str.strip().nunique()
    }

resumen_llaves = pd.DataFrame([
    auditar_llaves(tabla_modelo_2024, 2024),
    auditar_llaves(tabla_modelo_2025, 2025)
])

resumen_llaves

## 3. Perfil técnico de cada variable

Este perfil describe tipos detectados, cobertura, cardinalidad y ejemplos. No transforma ningún valor.

In [ ]:
def perfilar_variables(tabla, sufijo):
    filas = []

    for variable in tabla.columns:
        serie = tabla[variable]
        valores_no_nulos = serie.dropna()
        ejemplos = pd.unique(valores_no_nulos.astype('string'))[:3]

        filas.append({
            'variable': variable,
            f'tipo_{sufijo}': str(serie.dtype),
            f'no_nulos_{sufijo}': int(serie.notna().sum()),
            f'porcentaje_nulos_{sufijo}': serie.isna().mean(),
            f'valores_distintos_{sufijo}': int(serie.nunique(dropna=True)),
            f'ejemplos_{sufijo}': ' | '.join(map(str, ejemplos))
        })

    return pd.DataFrame(filas)

perfil_2024 = perfilar_variables(tabla_modelo_2024, '2024')
perfil_2025 = perfilar_variables(tabla_modelo_2025, '2025')

perfil_comparado = perfil_2024.merge(
    perfil_2025,
    on='variable',
    how='outer',
    validate='one_to_one'
)

perfil_comparado['tipos_coinciden'] = (
    perfil_comparado['tipo_2024'] == perfil_comparado['tipo_2025']
)

display(perfil_comparado.head())
print('Variables con tipos detectados distintos:')
display(perfil_comparado.loc[~perfil_comparado['tipos_coinciden'], [
    'variable', 'tipo_2024', 'tipo_2025',
    'ejemplos_2024', 'ejemplos_2025'
]])

## 4. Diccionario técnico inicial

El diccionario se genera a partir de la estructura actual. Las columnas `descripcion`, `rol_modelo` y `disponible_en_produccion` quedan como referencia para completar después de revisar la data; no se fija aún ningún target.

In [ ]:
# Columnas añadidas por cada módulo durante los notebooks de procesamiento.
COLUMNAS_1629 = {
    'n_residentes_habituales', 'n_menores_5_hogar',
    'n_ninos_6_17_hogar', 'n_adultos_18_59_hogar',
    'n_adultos_mayores_hogar'
}
COLUMNAS_1630 = {
    'quintil_riqueza', 'n_habitaciones_dormir',
    'personas_por_dormitorio'
}
COLUMNAS_1631 = {
    'edad_madre', 'nivel_educativo_madre', 'madre_afiliada_sis',
    'cuidador_habitual', 'lengua_materna'
}
COLUMNAS_1634 = {
    'tuvo_cred', 'cred_realizado_por_doctor',
    'cred_realizado_por_obstetra', 'cred_realizado_por_enfermera',
    'cred_realizado_por_tecnico_enfermeria',
    'cred_realizado_por_promotor_salud', 'cred_realizado_por_otro',
    'lugar_cred_codigo', 'n_cred_con_fecha', 'fecha_ultimo_cred',
    'n_entregas_mmn_registradas', 'registro_entrega_mmn',
    'n_entregas_mmn_con_fecha', 'fecha_ultima_entrega_mmn',
    'anio_entrevista', 'cmc_entrevista', 'cmc_ultimo_cred',
    'cmc_ultima_entrega_mmn', 'meses_desde_ultimo_cred',
    'meses_desde_ultima_entrega_mmn',
    'fecha_cred_posterior_entrevista',
    'fecha_entrega_mmn_posterior_entrevista'
}

def origen_variable(variable):
    if variable in COLUMNAS_1629:
        return 'Módulo 1629 - hogar'
    if variable in COLUMNAS_1630:
        return 'Módulo 1630 - vivienda'
    if variable in COLUMNAS_1631:
        return 'Módulo 1631 - madre/cuidadora'
    if variable in COLUMNAS_1634:
        return 'Módulo 1634 - niño/CRED'
    return 'Tabla niño procesada originalmente'

def nivel_variable(variable):
    if variable in COLUMNAS_1629 or variable in COLUMNAS_1630:
        return 'hogar'
    if variable in COLUMNAS_1631:
        return 'madre/cuidadora'
    return 'niño/entrevista'

def rol_preliminar(variable):
    if variable in {'caseid', 'hhid', 'idx_nino'}:
        return 'identificador - excluir del modelo'
    if variable in {'anio', 'peso_muestral', 'conglomerado', 'estrato'}:
        return 'diseño/validación - no predictor inicial'
    if variable in {'tomo_7d', 'no_tomo_7d', 'recibio_12m'}:
        return 'candidato a target - no usar como predictor'
    return 'por evaluar'

diccionario_tecnico = perfil_comparado.copy()
diccionario_tecnico['origen'] = diccionario_tecnico['variable'].map(origen_variable)
diccionario_tecnico['nivel'] = diccionario_tecnico['variable'].map(nivel_variable)
diccionario_tecnico['descripcion'] = pd.NA
diccionario_tecnico['rol_modelo'] = diccionario_tecnico['variable'].map(rol_preliminar)
diccionario_tecnico['disponible_en_produccion'] = 'por evaluar'

diccionario_tecnico = diccionario_tecnico[[
    'variable', 'origen', 'nivel', 'descripcion', 'rol_modelo',
    'tipo_2024', 'tipo_2025', 'tipos_coinciden',
    'no_nulos_2024', 'porcentaje_nulos_2024', 'valores_distintos_2024',
    'ejemplos_2024',
    'no_nulos_2025', 'porcentaje_nulos_2025', 'valores_distintos_2025',
    'ejemplos_2025', 'disponible_en_produccion'
]].sort_values(['origen', 'variable']).reset_index(drop=True)

diccionario_tecnico

## 5. Códigos especiales y faltantes

Los códigos como `8`, `98`, `998` o `9998` pueden significar `no sabe`, `no aplica` o inconsistencias según la pregunta. Esta sección solo los identifica; no los convierte automáticamente en nulos.

In [ ]:
CODIGOS_ESPECIALES = {
    '8', '9', '96', '97', '98', '99',
    '996', '997', '998', '999',
    '9996', '9997', '9998', '9999'
}

def auditar_codigos_especiales(tabla, anio):
    resultados = []

    for variable in tabla.columns:
        valores = tabla[variable].astype('string').str.strip()
        conteos = valores.value_counts(dropna=True)

        for codigo in CODIGOS_ESPECIALES:
            if codigo in conteos.index:
                resultados.append({
                    'anio': anio,
                    'variable': variable,
                    'codigo': codigo,
                    'n_registros': int(conteos[codigo]),
                    'porcentaje': conteos[codigo] / len(tabla)
                })

    return pd.DataFrame(resultados)

codigos_especiales = pd.concat([
    auditar_codigos_especiales(tabla_modelo_2024, 2024),
    auditar_codigos_especiales(tabla_modelo_2025, 2025)
], ignore_index=True)

codigos_especiales.sort_values(
    ['variable', 'anio', 'n_registros'],
    ascending=[True, True, False]
).head(50)

## 6. Unión bruta de los años

Esta unión conserva todos los valores tal como fueron cargados. Solo se ejecuta si las estructuras son idénticas. La tabla resultante permanece en memoria y aún no se guarda ni se usa para entrenar un modelo.

In [ ]:
solo_2024 = sorted(columnas_2024 - columnas_2025)
solo_2025 = sorted(columnas_2025 - columnas_2024)

if solo_2024 or solo_2025:
    raise ValueError(
        'No se unieron los años porque existen diferencias de columnas. '
        f'Solo 2024: {solo_2024}. Solo 2025: {solo_2025}'
    )

# Se preserva el orden de columnas del archivo 2024.
orden_columnas = tabla_modelo_2024.columns.tolist()

tabla_modelo_2024_2025_bruta = pd.concat(
    [
        tabla_modelo_2024[orden_columnas],
        tabla_modelo_2025[orden_columnas]
    ],
    ignore_index=True
)

duplicados_entre_anios = tabla_modelo_2024_2025_bruta.duplicated(
    ['anio', 'caseid', 'idx_nino']
).sum()

print(f'Filas unificadas: {len(tabla_modelo_2024_2025_bruta):,}')
print(f'Columnas: {tabla_modelo_2024_2025_bruta.shape[1]:,}')
print(f'Duplicados en anio + caseid + idx_nino: {duplicados_entre_anios:,}')

display(
    tabla_modelo_2024_2025_bruta['anio']
    .value_counts(dropna=False)
    .rename_axis('anio')
    .to_frame('n_ninos')
)

## Próximo paso

Revisar los resultados de este notebook y completar el diccionario técnico. Después se analizarán los posibles targets disponibles en la data, sin asumir todavía que `no_tomo_7d` será el target final.